# 10장 - RAG 벡터 DB 구축

원본 파일: `chap10/build_vectorstore.py`

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os

load_dotenv()

BASE_DIR = (os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd())
PDF_PATH = os.path.join(BASE_DIR, 'data', '2040_seoul_plan.pdf')
PERSIST_DIR = os.path.join(BASE_DIR, 'chroma_store')

# OpenRouter도 임베딩 엔드포인트를 지원하므로 동일한 클라이언트 설정 사용
embedding = OpenAIEmbeddings(
    model='text-embedding-3-large',
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv('OPENAI_API_KEY'),
)

/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def build():
    # 1) PDF 읽기
    loader = PyPDFLoader(PDF_PATH)
    data = loader.load()

    # 2) 청킹 (1000자 단위, 100자 오버랩)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = text_splitter.split_documents(data)
    print(f'총 청크 수: {len(splits)}')

    # 3) 오버랩 처리: 각 청크 뒤에 다음 청크의 앞부분을 덧붙여 문맥 단절 완화
    for i in range(len(splits) - 1):
        splits[i].page_content += "\n" + splits[i + 1].page_content[:100]

    # 4) 임베딩 후 크로마 DB에 저장 (한 번 만들어두면 재사용 가능)
    if not os.path.exists(PERSIST_DIR):
        print("Creating new Chroma store")
        vectorstore = Chroma.from_documents(
            documents=splits,
            embedding=embedding,
            persist_directory=PERSIST_DIR,
        )
    else:
        print("Loading existing Chroma store")
        vectorstore = Chroma(
            persist_directory=PERSIST_DIR,
            embedding_function=embedding,
        )

    return vectorstore

## 실행 / 테스트

In [3]:
vectorstore = build()

총 청크 수: 308
Loading existing Chroma store


In [4]:
retriever = vectorstore.as_retriever(k=3)
docs = retriever.invoke("서울시의 환경 정책에 대해 궁금해")

In [5]:
print("\n===== 유사 청크 검색 결과 =====")
for d in docs:
    print(d.page_content[:200])
    print('------')


===== 유사 청크 검색 결과 =====
제4절 기후·환경 부문1. 개요Ÿ기후변화는 21세기에 전 지구적으로 가장 위중한 영향을 미칠 것으로 예상되며, 시민 생활의 모든 측면과 연관되어 있어 향후 서울시의 적극적인 대응이 필요하다.Ÿ탄소중립 목표뿐만 아니라 미세먼지로부터 시민 건강을 지키기 위해서는 건물, 교통, 에너지 등 도시의 주요 인프라 전반의 혁신이 요구되며, 이를 위해 새로운 기술과 혁신
------
제4절 기후·환경 부문813-2-2 기후 행동 포용적 거버넌스 구축을 위한 시민 행동 활성화Ÿ자원순환부터 기후 행동까지 수도권 시민과 정부, 기업의 협력을 위해, 서울시-중앙부처-수도권이 함께 기후환경 관리를 위한 협력체계를 마련하고, 시민 정책참여 플랫폼과 민·관 협력형 사업 확대로 효과적인 규제·관리 기반을 강화한다.-개인·단체·기업 등 서울시 모든 구
------
Ÿ소규모 분산형 발전시설 확대를 위한 대체에너지 설치 지원을 지속적으로 확대하고, 공공시설의 옥상 및 주차장 등의 공간을 활용하여 지역 내에서 에너지를 생산·관리할 수 있는 기반을 마련한다.Ÿ지역에서 생산된 에너지 중 사용하고 남은 전력을 손쉽게 거래하여 순환 에너지 경제 기반이 확대될 수 있도록 법적·제도적으로 지원한다.3-1-4 대기 환경을 고려한 공간
------
78제3장 부문별 전략계획
제4절 기후·환경 부문1. 개요Ÿ기후변화는 21세기에 전 지구적으로 가장 위중한 영향을 미칠 것으로 예상되며, 시민 생활의 모든 측면과 연관되어 있어 향후 서울시의 적극적인 대응이
------
